# Bölüm 3: Olasılık — Uygulamalı Jupyter Notebook
**Haydar Kılıç | Yapay Zeka Mühendisliği**

Bu notebook, Bölüm 3'teki teorik kavramları Python ile uygulamalı olarak göstermektedir.

---
**İçindekiler:**
1. Rassal Süreçler ve Büyük Sayılar Yasası
2. Olasılık Kuralları (Toplama, Çarpma, Tümleyen)
3. Koşullu Olasılık ve Bağımsızlık
4. Bayes Teoremi ve Ağaç Diyagramları
5. Yerine Koyarak / Koymadan Örnekleme
6. Rassal Değişkenler: Beklenti ve Varyans
7. Sürekli Dağılımlar

In [ ]:
# Gerekli kütüphaneler
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import pandas as pd
from fractions import Fraction
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

# Grafik stili
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

np.random.seed(42)
print('Kütüphaneler yüklendi.')

---
## 1. Rassal Süreçler ve Büyük Sayılar Yasası

**Tanım:** Büyük sayılar yasası (BSY), gözlem sayısı arttıkça gözlemlenen oranın gerçek olasılığa yakınsadığını söyler:

$$\hat{p}_n \xrightarrow{n \to \infty} p$$

**Kumarbaz Yanılgısı:** Para, geçmişteki sonuçları "telafi etmek" zorunda değildir. Her atış birbirinden bağımsızdır.

In [ ]:
# ── Büyük Sayılar Yasası Simülasyonu ──
n_atış = 5000
atışlar = np.random.choice([0, 1], size=n_atış)  # 1 = Yazı
kümülatif_oran = np.cumsum(atışlar) / np.arange(1, n_atış + 1)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(kümülatif_oran, color='steelblue', lw=1.5, label='Gözlemlenen yazı oranı')
ax.axhline(0.5, color='tomato', lw=2, linestyle='--', label='Gerçek olasılık p = 0.5')
ax.fill_between(range(n_atış), kümülatif_oran, 0.5, alpha=0.15, color='steelblue')
ax.set_xlabel('Atış sayısı (n)')
ax.set_ylabel('Yazı oranı')
ax.set_title('Büyük Sayılar Yasası — Yazı-Tura Simülasyonu')
ax.legend()
ax.set_ylim(0.2, 0.8)
plt.tight_layout()
plt.show()

print(f'İlk 10 atışta yazı oranı : {kümülatif_oran[9]:.3f}')
print(f'İlk 100 atışta yazı oranı: {kümülatif_oran[99]:.3f}')
print(f'5000 atışta yazı oranı   : {kümülatif_oran[-1]:.3f}')

In [ ]:
# ── Kumarbaz Yanılgısı Gösterimi ──
# 10 kez üst üste yazı geldi — bir sonraki tura gelir mi?
# Her atış bağımsızdır, olasılık hâlâ 0.5!

n_deney = 100_000
# 10 yazının ardından gelen atışı simüle et
sonraki = np.random.choice([0, 1], size=n_deney)  # 1=Yazı, 0=Tura
yazı_oranı = sonraki.mean()

print('=== Kumarbaz Yanılgısı Testi ===')
print(f'10 kez yazı geldikten sonra {n_deney} deneylik simülasyon:')
print(f'  Yazı oranı : {yazı_oranı:.4f}  (Beklenen: 0.5000)')
print(f'  Tura oranı : {1-yazı_oranı:.4f}  (Beklenen: 0.5000)')
print()
print('Sonuç: Para geçmişi "hatırlamaz". Her atışta P(Yazı) = 0.5')

---
## 2. Olasılık Kuralları

### 2.1 Genel Toplama Kuralı

$$P(A \text{ veya } B) = P(A) + P(B) - P(A \text{ ve } B)$$

Ayrık olaylar için $P(A \text{ ve } B) = 0$, dolayısıyla:

$$P(A \text{ veya } B) = P(A) + P(B)$$

### 2.2 Tümleyen Kuralı
$$P(A^c) = 1 - P(A)$$

In [ ]:
# ── İskambil Destesi: Vale veya Kırmızı Kart ──
# P(vale veya kırmızı) = P(vale) + P(kırmızı) − P(vale ve kırmızı)

toplam_kart = 52
p_vale     = Fraction(4, toplam_kart)   # 4 vale
p_kırmızı  = Fraction(26, toplam_kart)  # 26 kırmızı kart
p_vale_kırmızı = Fraction(2, toplam_kart)  # 2 kırmızı vale

p_vale_veya_kırmızı = p_vale + p_kırmızı - p_vale_kırmızı

print('=== Vale veya Kırmızı Kart ===')
print(f'P(Vale)             = {p_vale} = {float(p_vale):.4f}')
print(f'P(Kırmızı)          = {p_kırmızı} = {float(p_kırmızı):.4f}')
print(f'P(Vale ve Kırmızı)  = {p_vale_kırmızı} = {float(p_vale_kırmızı):.4f}')
print(f'P(Vale veya Kırmızı)= {p_vale} + {p_kırmızı} - {p_vale_kırmızı} = {p_vale_veya_kırmızı} ≈ {float(p_vale_veya_kırmızı):.4f}')

In [ ]:
# ── Tümleyen Kuralı: En az 1 sigortasız Texas'lı ──
# P(en az 1 sigortasız) = 1 − P(hiç sigortasız yok)

p_sigortasız = 0.255
n = 5

p_hiç_sigortasız_yok = (1 - p_sigortasız) ** n
p_en_az_bir = 1 - p_hiç_sigortasız_yok

print('=== 5 Texas Sakini: En Az 1 Sigortasız ===')
print(f'P(sigortasız)             = {p_sigortasız}')
print(f'P(sigortalı)              = {1 - p_sigortasız}')
print(f'P(hiç sigortasız yok)     = {1-p_sigortasız}^5 = {p_hiç_sigortasız_yok:.4f}')
print(f'P(en az 1 sigortasız)     = 1 − {p_hiç_sigortasız_yok:.4f} = {p_en_az_bir:.4f}')

# Görselleştirme: n'ye göre değişim
ns = np.arange(1, 21)
probs = 1 - (1 - p_sigortasız) ** ns

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(ns, probs, color='steelblue', edgecolor='white', alpha=0.85)
ax.axhline(0.95, color='tomato', linestyle='--', lw=1.5, label='0.95 eşiği')
ax.set_xlabel('Örneklem büyüklüğü (n)')
ax.set_ylabel('P(en az 1 sigortasız)')
ax.set_title('Tümleyen Kuralı — Örneklem Büyüklüğüne Göre Olasılık')
ax.legend()
plt.tight_layout()
plt.show()

n_95 = next(i for i, p in enumerate(probs, 1) if p >= 0.95)
print(f'\nEn az 1 sigortasız olasılığının ≥ 0.95 olması için gereken n: {n_95}')

---
## 3. Koşullu Olasılık ve Bağımsızlık

$$P(A|B) = \frac{P(A \text{ ve } B)}{P(B)}$$

**Bağımsızlık testi:** $P(A|B) = P(A)$ ise A ve B bağımsızdır.

In [ ]:
# ── Nüks Çalışması: Koşullu Olasılık ──
veri = pd.DataFrame({
    'Tedavi'   : ['Desipramin', 'Desipramin', 'Lityum', 'Lityum', 'Plasebo', 'Plasebo'],
    'Durum'    : ['Nüks', 'Nüks Yok', 'Nüks', 'Nüks Yok', 'Nüks', 'Nüks Yok'],
    'Sayı'     : [10, 14, 18, 6, 20, 4]
})

tablo = veri.pivot(index='Tedavi', columns='Durum', values='Sayı')
tablo['Toplam'] = tablo.sum(axis=1)
tablo.loc['Toplam'] = tablo.sum()
print('=== Nüks Çalışması Çapraz Tablo ===')
print(tablo)

toplam = 72
print()
# Marjinal olasılık
p_nuks = 48/72
print(f'P(Nüks)                  = 48/72 = {p_nuks:.4f}')

# Birleşik olasılık
p_des_ve_nuks = 10/72
print(f'P(Desipramin ve Nüks)    = 10/72 = {p_des_ve_nuks:.4f}')

# Koşullu olasılıklar
p_nuks_des = 10/24
p_nuks_lit = 18/24
p_nuks_pla = 20/24
print(f'\nP(Nüks | Desipramin)     = 10/24 = {p_nuks_des:.4f}')
print(f'P(Nüks | Lityum)         = 18/24 = {p_nuks_lit:.4f}')
print(f'P(Nüks | Plasebo)        = 20/24 = {p_nuks_pla:.4f}')

In [ ]:
# ── Görselleştirme: Tedaviye Göre Nüks Oranları ──
tedaviler = ['Desipramin', 'Lityum', 'Plasebo']
nuks_oranı    = [10/24, 18/24, 20/24]
nuks_yok_oranı = [14/24, 6/24, 4/24]

x = np.arange(len(tedaviler))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, nuks_oranı, w, label='Nüks', color='tomato', alpha=0.85)
b2 = ax.bar(x + w/2, nuks_yok_oranı, w, label='Nüks Yok', color='steelblue', alpha=0.85)

ax.axhline(48/72, color='gray', linestyle='--', lw=1.5,
           label=f'Marjinal P(Nüks) = {48/72:.2f}')
ax.set_xticks(x)
ax.set_xticklabels(tedaviler)
ax.set_ylabel('Koşullu Olasılık')
ax.set_title('Tedaviye Göre Koşullu Nüks Olasılıkları')
ax.legend()

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── Bağımsızlık Testi: Silah Sahipliği & Irk ──
# Eğer P(A|B) = P(A) ise bağımsız

p_genel   = 0.58
p_beyaz   = 0.67
p_siyah   = 0.28
p_ispanyol = 0.64

print('=== Silah Sahipliği Görüşü ve Irk Bağımsızlık Testi ===')
print(f'P(Korur) [genel]           = {p_genel}')
print(f'P(Korur | Beyaz)           = {p_beyaz}')
print(f'P(Korur | Siyah)           = {p_siyah}')
print(f'P(Korur | İspanyol Kökenli)= {p_ispanyol}')
print()
print('Koşullu olasılıklar birbirinden ve P(Korur) genelinden farklı.')
print('→ Görüş ve ırk/etnik köken büyük olasılıkla BAĞIMLIDIR.')

---
## 4. Bayes Teoremi ve Olasılık Ağaçları

$$P(A_1|B) = \frac{P(B|A_1)\,P(A_1)}{P(B|A_1)\,P(A_1) + P(B|A_2)\,P(A_2) + \cdots + P(B|A_k)\,P(A_k)}$$

In [ ]:
# ── Meme Kanseri Taraması ──
def bayes_guncelle(p_kanser, duyarlılık=0.78, yanlış_pozitif=0.10):
    """Pozitif mammografi sonucu verilen kanser olasılığını hesaplar."""
    p_yok_kanser = 1 - p_kanser
    # P(+ | kanser) × P(kanser)
    p_pozitif_kanser = duyarlılık * p_kanser
    # P(+ | kanser yok) × P(kanser yok)
    p_pozitif_yok    = yanlış_pozitif * p_yok_kanser
    # Bayes formülü
    p_kanser_pozitif = p_pozitif_kanser / (p_pozitif_kanser + p_pozitif_yok)
    return p_kanser_pozitif, p_pozitif_kanser, p_pozitif_yok

# İlk test
p_ön = 0.017
p_sonraki, p_poz_kanser, p_poz_yok = bayes_guncelle(p_ön)

print('=== Mamografi — 1. Test ===')
print(f'Ön olasılık P(Kanser)          = {p_ön}')
print(f'P(+ | Kanser) × P(Kanser)      = {0.78}×{p_ön} = {p_poz_kanser:.4f}')
print(f'P(+ | Kanser Yok) × P(Yok)     = {0.10}×{1-p_ön:.3f} = {p_poz_yok:.4f}')
print(f'P(Kanser | +)                  = {p_poz_kanser:.4f}/{p_poz_kanser+p_poz_yok:.4f} = {p_sonraki:.4f}')

# İkinci test (güncellenen prior ile)
p_ön2 = p_sonraki
p_sonraki2, p_poz2, p_yok2 = bayes_guncelle(p_ön2)
print()
print('=== Mamografi — 2. Test (pozitif sonuçla güncellenen prior) ===')
print(f'Güncel prior P(Kanser)         = {p_ön2:.4f}')
print(f'P(Kanser | +) [2. test]        = {p_sonraki2:.4f}')

In [ ]:
# ── Bayes Güncellemesinin Görselleştirilmesi ──
priorlar = np.linspace(0.001, 0.5, 200)
posteriorlar = [bayes_guncelle(p)[0] for p in priorlar]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(priorlar, posteriorlar, color='steelblue', lw=2)
ax.plot([0, 0.5], [0, 0.5], 'gray', linestyle=':', lw=1.5, label='Prior = Posterior (bağımsız test)')

# İki testi işaretle
ax.scatter([0.017, p_sonraki], [p_sonraki, p_sonraki2],
           s=100, zorder=5, color='tomato',
           label=f'1. test: prior={0.017}, posterior={p_sonraki:.3f}\n'
                 f'2. test: prior={p_sonraki:.3f}, posterior={p_sonraki2:.3f}')

ax.set_xlabel('Ön Olasılık P(Kanser)')
ax.set_ylabel('Sonsal Olasılık P(Kanser | +)')
ax.set_title('Bayes Güncellemesi — Mammografi Örneği')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── SIR Modeli: Uygulama Etkinliği ──
# Nüfus dağılımı: Duyarlı %60, Enfekte %10, İyileşmiş %30
# Test doğruluğu: D için %95 neg, E için %99 poz, İ için %65 neg

p_d, p_e, p_i = 0.60, 0.10, 0.30

# Her grup için P(pozitif)
p_poz_d = 0.05   # duyarlı → yanlış pozitif
p_poz_e = 0.99   # enfekte → doğru pozitif
p_poz_i = 0.35   # iyileşmiş → yanlış pozitif

# Birleşik olasılıklar
p_d_poz = p_d * p_poz_d
p_e_poz = p_e * p_poz_e
p_i_poz = p_i * p_poz_i
p_poz_toplam = p_d_poz + p_e_poz + p_i_poz

# Bayes
p_enf_poz = p_e_poz / p_poz_toplam

print('=== SIR Modeli — Bayes Teoremi ===')
print(f'P(Duyarlı)  × P(+ | Duyarlı)   = {p_d}×{p_poz_d} = {p_d_poz:.4f}')
print(f'P(Enfekte)  × P(+ | Enfekte)   = {p_e}×{p_poz_e} = {p_e_poz:.4f}')
print(f'P(İyileşmiş)× P(+ | İyileşmiş) = {p_i}×{p_poz_i} = {p_i_poz:.4f}')
print(f'P(+) [toplam]                  = {p_poz_toplam:.4f}')
print(f'P(Enfekte | +)                 = {p_e_poz:.4f}/{p_poz_toplam:.4f} = {p_enf_poz:.4f}')

# Pasta grafiği
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

axs[0].pie([p_d, p_e, p_i], labels=['Duyarlı', 'Enfekte', 'İyileşmiş'],
           autopct='%1.0f%%', colors=['#5B9BD5', '#ED7D31', '#70AD47'],
           startangle=90)
axs[0].set_title('Nüfus Dağılımı')

axs[1].pie([p_d_poz, p_e_poz, p_i_poz],
           labels=[f'Duyarlı+\n({p_d_poz:.3f})',
                   f'Enfekte+\n({p_e_poz:.3f})',
                   f'İyileşmiş+\n({p_i_poz:.3f})'],
           autopct='%1.1f%%', colors=['#5B9BD5', '#ED7D31', '#70AD47'],
           startangle=90)
axs[1].set_title(f'Pozitif Test Verenler\nP(Enfekte|+) = {p_enf_poz:.3f}')

plt.suptitle('SIR Modeli — Bayes Analizi', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Yerine Koyarak / Koymadan Örnekleme

| | Yerine Koyarak | Yerine Koymadan |
|---|---|---|
| Torba kompozisyonu | Değişmez | Değişir |
| Çekişler | Bağımsız | Bağımlı |

In [ ]:
# ── Fiş Torbası Simülasyonu ──
# 5 kırmızı, 3 mavi, 2 turuncu
torba = ['K']*5 + ['M']*3 + ['T']*2
print(f'Torba: {torba}  (toplam {len(torba)} fiş)')

n_deney = 100_000

# Yerine koyarak: 2 mavi
yk_sayım = 0
for _ in range(n_deney):
    c1 = np.random.choice(torba)
    c2 = np.random.choice(torba)       # torba değişmedi
    if c1 == 'M' and c2 == 'M':
        yk_sayım += 1

# Yerine koymadan: 2 mavi
ykm_sayım = 0
for _ in range(n_deney):
    kalan = torba.copy()
    c1 = np.random.choice(kalan)
    kalan.remove(c1)
    c2 = np.random.choice(kalan)       # torba değişti
    if c1 == 'M' and c2 == 'M':
        ykm_sayım += 1

print()
print('=== Arka Arkaya 2 Mavi Fiş Çekme ===')
print(f'Teorik (YK)  : P(M)×P(M)     = 0.3×0.3 = 0.09')
print(f'Simüle (YK)  :                           = {yk_sayım/n_deney:.4f}')
print()
print(f'Teorik (YKM) : P(M)×P(M|M)   = 0.3×0.222 = {0.3*2/9:.4f}')
print(f'Simüle (YKM) :                            = {ykm_sayım/n_deney:.4f}')

In [ ]:
# ── İskambil: As Sonra 3 (Yerine Koymadan) ──
p_as = Fraction(4, 52)
p_uc_as_sonra = Fraction(4, 51)   # as çekildikten sonra kalan
p_as_sonra_3 = p_as * p_uc_as_sonra

print('=== As Sonra 3 (Yerine Koymadan) ===')
print(f'P(As)              = 4/52 = {float(p_as):.4f}')
print(f'P(3 | As çekildi)  = 4/51 = {float(p_uc_as_sonra):.4f}')
print(f'P(As sonra 3)      = {p_as_sonra_3} ≈ {float(p_as_sonra_3):.4f}')

---
## 6. Rassal Değişkenler: Beklenti ve Varyans

$$E(X) = \mu = \sum_{i=1}^{k} x_i \, P(X = x_i)$$

$$\text{Var}(X) = \sigma^2 = \sum_{i=1}^{k} (x_i - E(X))^2 \, P(X = x_i)$$

In [ ]:
# ── Kart Oyunu: Beklenti ve Varyans ──
# Kupa (as değil) → 1 TL | As → 5 TL | Maça kızı → 10 TL | Diğer → 0

kazanç = np.array([1,    5,   10,   0  ])
olas   = np.array([12/52, 4/52, 1/52, 35/52])
etiket = ['Kupa\n(as değil)', 'As', 'Maça\nKızı', 'Diğer']

# Beklenti
E_X = np.sum(kazanç * olas)

# Varyans
Var_X = np.sum((kazanç - E_X)**2 * olas)
SS_X  = np.sqrt(Var_X)

# Tablo
df = pd.DataFrame({'Olay': etiket,
                   'Kazanç (TL)': kazanç,
                   'P(X)': [f'{o:.4f}' for o in olas],
                   'X·P(X)': kazanç * olas,
                   '(X-E[X])²·P(X)': (kazanç - E_X)**2 * olas})
print(df.to_string(index=False))
print(f'\nE(X)   = {E_X:.4f} TL')
print(f'Var(X) = {Var_X:.4f}')
print(f'SS(X)  = {SS_X:.4f} TL')

In [ ]:
# ── Olasılık Dağılımı Grafiği ──
fig, ax = plt.subplots(figsize=(10, 5))
renkler = ['steelblue', 'steelblue', 'steelblue', 'lightgray']
ax.bar(kazanç, olas, width=0.6, color=renkler, edgecolor='white')
ax.axvline(E_X, color='tomato', lw=2, linestyle='--', label=f'E(X) = {E_X:.2f} TL')
ax.set_xlabel('Kazanç (TL)')
ax.set_ylabel('Olasılık P(X)')
ax.set_title('Kart Oyunu — Kazanç Olasılık Dağılımı')
ax.set_xticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Doğrusal Kombinasyonlar: Ödev Süresi ──
# E(5İ + 4K) = 5·E(İ) + 4·E(K)
# Var(5İ + 4K) = 5·Var(İ) + 4·Var(K)  [bağımsızlık varsayımı]

E_ist   = 10   # dk
E_kim   = 15   # dk
SS_ist  = 1.5  # dk
SS_kim  = 2.0  # dk

n_ist, n_kim = 5, 4

E_toplam   = n_ist * E_ist + n_kim * E_kim
Var_toplam = n_ist * SS_ist**2 + n_kim * SS_kim**2
SS_toplam  = np.sqrt(Var_toplam)

print('=== Haftalık Ödev Süresi ===')
print(f'E(5·İst + 4·Kim) = 5×{E_ist} + 4×{E_kim} = {E_toplam} dakika')
print(f'Var(5·İst + 4·Kim) = 5×{SS_ist}² + 4×{SS_kim}² = {Var_toplam}')
print(f'SS(Toplam) = √{Var_toplam} = {SS_toplam:.4f} dakika')

print()
print('=== X+X ile 2X Farkı ===')
E_XX  = 2 * E_ist
E_2X  = 2 * E_ist
Var_XX = 2 * SS_ist**2
Var_2X = 4 * SS_ist**2
print(f'E(X+X) = {E_XX}  =  E(2X) = {E_2X}   → Beklentiler eşit')
print(f'Var(X+X) = 2×{SS_ist}² = {Var_XX}  ≠  Var(2X) = 4×{SS_ist}² = {Var_2X}   → Varyanslar FARKLI!')

In [ ]:
# ── Kumarhane Oyunu: Adil Oyun Testi ──
# Giriş 5 TL. İlk kart kırmızı ise ikinci kart çekiliyor.
# Maça ası ise 500 TL, yoksa 0.

maliyet = 5
p_kırmızı_maça_as = (26/52) * (1/51)   # kırmızı kart çekip ardından maça ası çekme
p_diğer = 1 - p_kırmızı_maça_as

kâr_kazan = 500 - maliyet
kâr_kaybet = 0 - maliyet

E_kâr = kâr_kazan * p_kırmızı_maça_as + kâr_kaybet * p_diğer

print('=== Kumarhane Oyunu ===')
print(f'P(Kırmızı ve Maça Ası) = 26/52 × 1/51 = {p_kırmızı_maça_as:.4f}')
print(f'Kazanma kârı           = 500 - 5 = {kâr_kazan} TL')
print(f'Kaybetme kârı          = 0 - 5   = {kâr_kaybet} TL')
print(f'E(Kâr)                 = {kâr_kazan}×{p_kırmızı_maça_as:.4f} + ({kâr_kaybet})×{p_diğer:.4f}')
print(f'                       = {E_kâr:.4f} TL  (≈ {E_kâr*100:.1f} kuruş kayıp)')
print()
print('Bu oyun oyuncu lehine değildir — beklenen kayıp vardır.')
print('Adil oyun olması için: E(Kâr) = 0  →  Giriş ücreti ≈ {:.2f} TL olmalı.'.format(
    500 * p_kırmızı_maça_as))

---
## 7. Sürekli Dağılımlar

Sürekli rassal değişkenlerde olasılık, olasılık yoğunluk fonksiyonunun **eğrinin altındaki alanı** ile ölçülür:

$$P(a \leq X \leq b) = \int_a^b f(x)\,dx$$

Tek bir noktanın olasılığı her zaman sıfırdır: $P(X = c) = 0$

In [ ]:
# ── Sürekli Dağılım: ABD'li Yetişkinlerin Boyu ──
# Normal dağılım: μ = 170 cm, σ = 10 cm (yaklaşık)

mu, sigma = 170, 10
x = np.linspace(130, 210, 500)
pdf = stats.norm.pdf(x, mu, sigma)

# P(180 ≤ X ≤ 185)
a, b = 180, 185
p_aralik = stats.norm.cdf(b, mu, sigma) - stats.norm.cdf(a, mu, sigma)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax in axes:
    ax.plot(x, pdf, 'steelblue', lw=2.5)
    ax.fill_between(x, pdf, alpha=0.08, color='steelblue')
    ax.set_xlabel('Boy (cm)')
    ax.set_ylabel('Olasılık Yoğunluğu')

# Sol: 180–185 aralığı
xf = np.linspace(a, b, 300)
axes[0].fill_between(xf, stats.norm.pdf(xf, mu, sigma),
                     color='steelblue', alpha=0.7)
axes[0].set_title(f'P(180 ≤ X ≤ 185) = {p_aralik:.4f}')

# Sağ: Tam 180 cm — olasılık 0
axes[1].axvline(180, color='tomato', lw=2.5, label='X = 180')
axes[1].set_title('P(X = 180) = 0  (tek nokta, alan yok)')
axes[1].legend()

plt.suptitle('Sürekli Dağılım — Boy Örneği', fontsize=14)
plt.tight_layout()
plt.show()

print(f'P(180 ≤ Boy ≤ 185) = {p_aralik:.4f}')
print(f'P(Boy = 180)       = 0  (tanım gereği)')

In [ ]:
# ── Farklı Aralıklar İçin Olasılıklar ──
aralıklar = [(155, 165), (165, 175), (175, 185), (185, 195)]
renkler   = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(x, pdf, 'black', lw=2)

for (a_, b_), renk in zip(aralıklar, renkler):
    xf = np.linspace(a_, b_, 200)
    p  = stats.norm.cdf(b_, mu, sigma) - stats.norm.cdf(a_, mu, sigma)
    ax.fill_between(xf, stats.norm.pdf(xf, mu, sigma),
                    color=renk, alpha=0.55,
                    label=f'P({a_}–{b_}) = {p:.3f}')

ax.set_xlabel('Boy (cm)')
ax.set_ylabel('Yoğunluk')
ax.set_title('Normal Dağılım — Farklı Aralıkların Olasılıkları')
ax.legend()
plt.tight_layout()
plt.show()

---
## Özet ve Formüller

| Kural | Formül |
|---|---|
| Genel Toplama | $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ |
| Tümleyen | $P(A^c) = 1 - P(A)$ |
| Koşullu Olasılık | $P(A|B) = P(A \cap B)/P(B)$ |
| Bağımsız Olaylar | $P(A \cap B) = P(A) \times P(B)$ |
| Genel Çarpım | $P(A \cap B) = P(A|B) \times P(B)$ |
| Bayes Teoremi | $P(A_1|B) = \frac{P(B|A_1)P(A_1)}{\sum_i P(B|A_i)P(A_i)}$ |
| Beklenti | $E(X) = \sum x_i P(X=x_i)$ |
| Varyans | $\text{Var}(X) = \sum (x_i - E(X))^2 P(X=x_i)$ |
| Doğrusal Komb. | $E(aX+bY) = aE(X)+bE(Y)$ |
| Bağ. Var. | $\text{Var}(aX+bY) = a^2\text{Var}(X)+b^2\text{Var}(Y)$ |